# 06 — Expérience historique de copie directionnelle

> Ceci est une expérience historique, pas une stratégie live. Les entrées sont exécutées avec latence, réservées en capital et réglées à la date officielle.

In [ ]:
import polars as pl

from poly_data.analysis.punter import entries_only, punter_position_timeline, punter_view, simulate_copy_bet, simulate_random_cohort
from poly_data.io.parquet_store import ParquetStore
from poly_data.notebooks import resolve_notebook_context, source_inventory

ctx = resolve_notebook_context(); store = ParquetStore(ctx.root)
print({'root': str(ctx.root), 'mode': ctx.mode, 'revision': ctx.revision})
source_inventory(store, ['trades', 'market_outcomes'])

In [ ]:
punters = punter_view(store.scan('trades'), price_floor=0.02, price_ceiling=0.98).collect()
entries = entries_only(punter_position_timeline(punters))
outcomes = store.scan('market_outcomes').select(['market_id', 'winner_token', 'resolved_at']).collect()
cutoff = int(entries['timestamp'].quantile(0.7))
leaders = set(entries.filter(pl.col('timestamp') < cutoff).group_by('taker').len().sort('len', descending=True).head(5)['taker'].to_list())
print({'entries': entries.height, 'leaders': len(leaders), 'cutoff': cutoff})

In [ ]:
result = simulate_copy_bet(entries, punters, outcomes, leaders, train_end_ts=cutoff, test_end_ts=int(entries['timestamp'].max()) + 1, latency_secs=60, fee_bps=10, random_seed=7)
result.bets

In [ ]:
control = simulate_random_cohort(entries.filter(pl.col('timestamp') < cutoff), seed=7, n_leaders=len(leaders))
print({'random_leaders': sorted(control.leaders), **result.summary})
result.cashflows.sort('timestamp')

Les flux négatifs arrivent à l'exécution ; les flux positifs n'arrivent qu'au règlement officiel. Les marchés non réglés restent du capital réservé, et les comparaisons de cohortes aléatoires doivent être répétées sur plusieurs seeds.